# C++ Emergency Job Scheduler — Learning Journal

**Author:** Ali

**Progress documented through:** 1 October 2026

This is my personal record of building a small C++ job scheduler with guided feedback. I started unsure where to begin and learned by writing code, questioning the design, and correcting my own attempts. The exercise gave me a practical way to work on classes, encapsulation, vectors, const, and STL algorithms.

C++ snippets are displayed in Markdown so this notebook opens in standard JupyterLab without requiring a C++ kernel. The examples document the code and reasoning from the conversation, rather than recorded execution results.

# 1. Understanding the assignment

The scheduler manages jobs with an ID, description, priority, and duration. Priority ranges from 1 to 5; 5 is most urgent. Required operations: add, cancel, find, select next job, calculate total estimated time, and list jobs by priority.

Constraints include an initial `std::vector`, `std::string`, appropriate const and references, two STL algorithms, valid state, no raw `new`/`delete`, and protected access to the container. This journal focuses on the scheduler code we worked through together.

My learning approach was to separate responsibilities, define method behavior, implement one operation at a time, and trace small examples before moving forward.

# 2. From one class to two responsibilities

The first header put job fields and collection operations in one `JOB` class, with a vector outside it. The next attempt used inheritance: `class JobManager : JOB`.

We changed to composition because a manager **has jobs**. It is not a specialized job.

| Class | Responsibility |
|---|---|
| `JOB` | Represent one job and provide getters |
| `JobManager` | Own and manage the collection |

`getId`, `getDescription`, `getPriority`, and `getDuration` belong to `JOB`. Collection operations belong to `JobManager`. The final shared header moved all job fields into `private` and kept `std::vector<JOB> jobs` private in the manager. `#pragma once` was added.

# 3. Encapsulation and access

A private vector prevents callers from adding, erasing, or rearranging jobs directly. A private job ID prevents direct external changes to that field. Storing objects in a vector does not grant access to their private members: the manager uses public getters.

```cpp
jobs[i].getId()
```

This was the key distinction: storing an object and accessing its private data are different things. This made encapsulation more concrete for me: private data helps a class control how its state changes.

# 4. Constructor validation and initialization

The constructor uses a member initializer list and then validates the inputs. The implemented checks reject priority outside 1–5, duration at or below zero, negative IDs, empty descriptions, and descriptions over 100 characters.

The ID and description restrictions are our implementation choices; the original statement only explicitly specifies the priority range. These were extra choices in my implementation, rather than requirements explicitly stated in the assignment.

```cpp
JOB::JOB(int id, std::string description, int priority, int duration)
    : id(id), description(description), priority(priority), duration(duration) {
    // Input validation follows.
}
```

We removed repeated assignments from the final `else`: the initializer list already initializes those members. `std::invalid_argument` requires `<stdexcept>`. If validation throws, construction fails and a successfully constructed invalid job is not produced.

# 5. Finding and adding jobs

The first search returned the matching ID. We clarified the contract: `findJob(id)` returns the **current vector index**, or `-1` if absent. An ID is identity; an index is location.

For IDs `[42, 7, 100]`, searching for 7 returns index 1. The failure return belongs after the entire search, not after the first mismatch.

`addJob` checks for duplicates using `findJob`, then constructs the job with `emplace_back`. Duplicate IDs cause an exception. Constructor validation checks the remaining inputs.

Indexes are temporary: sorting and erasing can change them. Never retain a returned index as a permanent job identifier. I also learned that `jobs.size()` uses an unsigned size type, which explains why an `int` loop counter can cause a compiler warning.

# 6. Selecting the next job: the debugging path

The early implementation returned inside the loop and ran an additional printing loop during the search. This could stop at the first improvement, miss a later priority 5, repeat output, or reach the end without returning.

We introduced a bookmark: `highestPriorityIndex` records the most urgent job seen so far. Each strictly higher priority replaces the bookmark. The return happens after all jobs have been checked.

Duplicate priorities are allowed; only IDs are unique. Using strict `>` preserves the earliest job in vector order among ties. With stable sorting and the current add/cancel operations, that preserves first-added order among tied jobs.

I chose to print all jobs sharing the maximum priority as well as return one selected job ID. The printing pass now happens **after** determining the final maximum. The method leaves jobs pending; it does not execute or remove them. Repeated calls without changing the collection select the same job.

An empty collection throws `std::runtime_error`. The latest implementation returns the selected ID, not its priority.

# 7. What const and references mean here

A trailing `const` states that this method does not modify the ordinary stored members of the object. Reading and printing are compatible with it. Therefore `nextJob() const`, getters, `findJob() const`, and `totalDuration() const` fit the current behavior. The header and definition must agree.

Sorting the internal vector changes the object, so `jobsbyPriority()` is non-const.

`const JOB&` gives read-only access to an existing job without copying it. We used it in loops and the sorting comparator. My `getDescription()` returns a string by value. Seeing this beside `const JOB&` helped distinguish returning a copy from borrowing access to an existing object.

# 8. Cancellation and total duration

`cancelJob` obtains a fresh index from `findJob`, throws if the job is absent, and erases that element.

```cpp
jobs.erase(jobs.begin() + index);
```

`jobs.begin() + index` is an iterator, not an ID. Erasure shifts subsequent positions; sorting likewise changes locations.

`totalDuration()` currently loops over jobs by const reference and adds their durations to an integer total. The loop made the accumulation process explicit: start at zero, visit each job, and add its duration.

# 9. Learning stable_sort and lambdas

I first considered the plain-array pattern `sizeof(arr) / sizeof(arr[0])`. For a vector, iterators describe the range: `jobs.begin()` and `jobs.end()`.

`std::stable_sort` accepts a comparator as its third argument. A lambda supplies that callable inline; a named function would also work. Sorting chooses the pairs to compare, while we supply only the rule.

```cpp
std::stable_sort(jobs.begin(), jobs.end(),
    [](const JOB& a, const JOB& b) {
        return a.getPriority() > b.getPriority();
    });
```

| Part | Meaning |
|---|---|
| `[]` | No surrounding variables are captured |
| `const JOB& a, const JOB& b` | Read two jobs without copying |
| `a.getPriority() > b.getPriority()` | Higher priority goes first |

We fixed the missing `const` on the second parameter and the missing semicolon inside the lambda. Strict `>` also returns false for equal priorities, allowing stable sorting to preserve their relative order. `<algorithm>` provides the sorting algorithm.

# 10. Printing the sorted collection

The first `printf` attempt passed a `std::string` to `%s` and supplied fewer arguments than placeholders. That was undefined behavior. We switched to stream output, which accepts `std::string` directly.

The latest method shared in the conversation is:

```cpp
void JobManager::jobsbyPriority() {
    std::stable_sort(jobs.begin(), jobs.end(),
        [](const JOB& a, const JOB& b) {
            return a.getPriority() > b.getPriority();
        });
    int index = 1;
    for (const auto& job : jobs) {
        std::cout << index << " : " << job.getDescription()
            << "\n Job ID :< " << job.getId() << " > | Duration: "
            << job.getDuration() << " | Priority: "
            << job.getPriority() << "\n";
        index++;
    }
}
```

The printed number starts at 1 and is a display row number, not the zero-based vector index. Using `std::cout` here felt more natural because it accepts strings and numbers directly without matching them to format placeholders.

# 11. What this project taught me

The most important part of this project was learning to explain what each piece of code was responsible for. At first, I placed job data and collection operations in the same class. Separating them helped me understand why a manager contains jobs and how public methods provide controlled access to private data.

The distinction between an ID and an index was another turning point. I initially returned the job ID from `findJob`, then realized that cancellation needed its current position in the vector. That made the return value a design decision rather than simply an integer to send back.

I spent the most time on `nextJob`. I wanted to show all equally urgent jobs and questioned why only one job should be selected. Working through that confusion clarified two different actions: identifying one next job and displaying all jobs tied at the highest priority. My implementation does both, using a complete search followed by a separate printing pass.

Moving the return statement outside the loop was a small edit with a large effect. It taught me to ask whether I had examined all the information needed before deciding on a result. Tracing priorities such as `[2, 3, 5]` made that easier to see.

Lambdas were unfamiliar at first. Understanding that `stable_sort` asks me for a comparison rule made the syntax less mysterious. I did not need to write the sorting process myself; I needed to express when one job should come before another. Using `const JOB&` connected that new syntax to the ideas of read-only access and avoiding copies.

By this point, my code represented jobs, checked inputs, rejected duplicate IDs, searched and cancelled jobs, calculated total duration, selected a highest-priority job, and sorted and displayed the collection. The value of the exercise was the reasoning behind those operations: I could connect the class structure, loop behavior, and ordering rules to the behavior I wanted.
